# Bar plots and tables  of costs

## Import packages

In [ ]:
import time
from datetime import datetime
import os
import json
import yaml
import numpy as np 
import pandas as pd 
import pyarrow as pa
import glob
import xarray as xr
import random
import joblib
import matplotlib.pyplot as plt

from src import figure_ops
from src import input_ops
from src import df_ops
from src import file_ops

## Define functions

In [ ]:
# --- Helper functions ---
def below_threshold(column_data, threshold):
    return column_data < threshold
def in_threshold_range(column_data, threshold1, threshold2):
    return (column_data >= threshold1) & (column_data < threshold2)
def above_threshold(column_data, threshold):
    return column_data >= threshold

## --- Add KVA column to xfer/lines dataframe using (Amps * kV)---
def mva_series_lines(df):
    if "NormAmps [A]" not in df.columns or "Nominal V [kV]" not in df.columns:
        return pd.Series(np.zeros(len(df)), index=df.index, dtype=float)
    amps = pd.to_numeric(df["NormAmps [A]"], errors="coerce").fillna(0.0)
    kv   = pd.to_numeric(df["Nominal V [kV]"], errors="coerce").fillna(0.0)
    return (amps * kv) 


def get_upgrade_costs(grid_need_kw, upgrade_type="circuit"):
    """
    Returns upgrade cost statistics for circuit or substation based on grid need in kW.

    Parameters:
        grid_need_kw (float): The grid need in kilowatts.
        upgrade_type (str): 'circuit' or 'substation'.

    Returns:
        pd.Series: Selected upgrade cost statistics.
    """
    grid_need_mw = grid_need_kw / 1000  # Convert kW to MW

    # Determine row based on grid need
    if grid_need_mw < 1:
        row = cost_df.iloc[0]
    elif 1 <= grid_need_mw < 2:
        row = cost_df.iloc[1]
    elif 2 <= grid_need_mw < 4:
        row = cost_df.iloc[2]
    elif 4 <= grid_need_mw < 8:
        row = cost_df.iloc[3]
    else:  # grid_need_mw >= 8
        row = cost_df.iloc[4]

    # Filter based on upgrade type
    if upgrade_type == "circuit":
        return row.filter(like="Circuit upgrade")
    elif upgrade_type == "substation":
        return row.filter(like="Substation upgrade")
    else:
        raise ValueError("upgrade_type must be either 'circuit' or 'substation'")

        

# -----------------------------------------------------------
# Helper: histogram matching for LV assets
# -----------------------------------------------------------
def extract_min_kv(kv_entry):
    """
    Extract a representative kV value from the 'KV rating [kV]' column.

    The column may contain:
    - a list of kV values (one per winding)
    - a scalar (rare but possible)

    We use the minimum kV to classify LV / MV / HV transformers.
    """
    if isinstance(kv_entry, (list, tuple, np.ndarray)):
        return min(kv_entry)
    else:
        return float(kv_entry)
# -----------------------------------------------------------
# helper for line voltage (scalar kV)
# -----------------------------------------------------------
def extract_line_kv(kv_entry):
    """
    Extract kV value from 'Nominal V [kV]' for lines.

    The column is scalar (float), unlike transformers.
    """
    return float(kv_entry)

    
def assign_ages_deterministic(n_assets, age_df, seed=12345):
    """
    Deterministically assigns ages to n_assets such that the resulting
    age histogram exactly matches the empirical distribution (up to rounding).

    Parameters
    ----------
    n_assets : int
        Number of transformers to assign ages to.
    age_df : pd.DataFrame
        DataFrame with columns:
        - 'Age (Years)'
        - 'prob' (normalized to sum to 1)

    Returns
    -------
    np.ndarray
        Array of ages of length n_assets.
    """
    # Expected count per age
    expected_counts = age_df["prob"] * n_assets

    # Floor to integers
    counts = np.floor(expected_counts).astype(int)

    # Distribute remainder to ages with largest fractional parts
    remainder = n_assets - counts.sum()
    fractional = expected_counts - counts

    if remainder > 0:
        idx = np.argsort(fractional.values)[::-1][:remainder]
        counts.iloc[idx] += 1

    # Build the deterministic age vector
    ages_out = np.repeat(age_df["Age (Years)"].values, counts.values)
    
    # ---------------------------------------------------
    # reproducible random permutation
    # ---------------------------------------------------
    rng = np.random.default_rng(seed)
    rng.shuffle(ages_out)

    # Safety check
    assert len(ages_out) == n_assets, "Age assignment length mismatch"

    return ages_out
        

## Load config file with scenarios and parameters 

In [ ]:
config_file_name = 'opendss_config1'; config_path = f"config/{config_file_name}.yaml"; config = input_ops.load_config(config_path)

smart_ds_year = config['smart_ds_years'][0]

## Initialize parameters for saving paths
output_pf_path = config['output_pf_path']

with open(config_path, "r") as file:
    pf_config = yaml.safe_load(file)
    
# Percent-of-peak range to load (e.g., top 0–10% hours)
start_row_percent = config['start_row_percent']
top_percent_mdh = config['top_percent_mdh']

# File names
transformers_file_name = f"transformers_top_{start_row_percent}_{top_percent_mdh}_percent"
lines_file_name = f"lines_top_{start_row_percent}_{top_percent_mdh}_percent"

# Solar / battery SMART-DS scenario parameters
solar_share = config.get("solar_share", "none")
battery_share = config.get("battery_share", "none")

solar_battery_scenario_folder = input_ops.build_solar_battery_scenario_folder(
    solar_share=solar_share,
    battery_share=battery_share,
)

print(f"solar_share: {solar_share}\n battery_share: {battery_share}\n solar_battery_scenario_folder: {solar_battery_scenario_folder}")


print(f"\noutput_pf_path:{output_pf_path}")

## Set thresholds and loading columns, load customers and cost data, and AUX functions

In [ ]:
# Grid reinforcments Thresholds
xfm_cand=80 # transformers candidate overloading threshold
xfm_crit=100 # transformers critical overloading threshold
line_cand=67 # lines candidate overloading threshold
line_crit=100 # lines critical overloading threshold
  
# Unit-cost percentile columns used in the cost calculations
cost_med_col = 'Upgrade cost median ($/kW)' 
cost_25p_col = 'Upgrade cost 25th ($/kW)'  
cost_75p_col = 'Upgrade cost 75th ($/kW)'  


# Distribution-system upgrade unit costs from Salma et al. (utility project data)
customers_austin = {"P1R": 13894, "P1U": 65529, "P2U": 24279}
customers_gso    = {"Rural": 6115, "Urban-suburban": 61354, "Industrial": 3082}
customers_sf     = {"P1R": 48139, "P1U": 8896,  "P2U": 46378}


## Define distribution asset costs $/kW from Salma et al. (utility projects data)
cost_data = {
    "Grid need (MW)": ["<1", "1–<2", "2–<4", "4–<8", "≥8"],
    "Circuit upgrade 25th pct. ($/kW)": [445.71, 251.84, 196.76, 268.65, 237.23],
    "Circuit upgrade Median ($/kW)": [1875.00, 1368.89, 673.35, 438.14, 367.85],
    "Circuit upgrade 75th pct. ($/kW)": [5791.67, 2092.89, 1447.55, 785.30, 586.32],
    "Substation upgrade 25th pct. ($/kW)": [9935.00, 3594.99, 2927.93, 1137.16, 634.01],
    "Substation upgrade Median ($/kW)": [18863.43, 4740.03, 3978.38, 2004.70, 887.62],
    "Substation upgrade 75th pct. ($/kW)": [28982.79, 6980.09, 4468.75, 2570.87, 1189.14]
}

cost_df = pd.DataFrame(cost_data)
display(cost_df)


# -----------------------------------------------------------
#  empirical transformer age distribution
# -----------------------------------------------------------
# Data from NREL's transformer age distribution  
ages = list(range(64))
percentages = [
    0.00, 0.02, 0.05, 0.09, 0.14, 0.19, 0.28, 0.38, 0.49, 0.60,
    0.72, 0.85, 1.00, 1.15, 1.30, 1.45, 1.60, 1.75, 1.90, 2.05,
    2.18, 2.30, 2.42, 2.52, 2.61, 2.70, 2.78, 2.84, 2.90, 2.95,
    2.99, 3.04, 3.08, 3.12, 3.15, 3.18, 3.21, 3.22, 3.19, 3.12,
    3.00, 2.85, 2.65, 2.45, 2.25, 2.05, 1.85, 1.65, 1.45, 1.25,
    1.10, 0.95, 0.82, 0.70, 0.60, 0.52, 0.45, 0.38, 0.32, 0.28,
    0.24, 0.20, 0.17, 0.14
]

age_df = pd.DataFrame({
    "Age (Years)": ages,
    "Percentage (%)": percentages
})

# Normalize percentages to a probability mass function
age_df["prob"] = age_df["Percentage (%)"] / age_df["Percentage (%)"].sum()

# Median age (used for MV / HV transformers)
median_age = (
    age_df
    .loc[age_df["prob"].cumsum() >= 0.5, "Age (Years)"]
    .iloc[0]
)
display(age_df)
        
# Example usage
print(get_upgrade_costs(1500, upgrade_type="circuit").values)
print(get_upgrade_costs(1500, upgrade_type="substation").values)

## Set parameters

In [ ]:
hist_col = "median_annual_max_loading_historical_1990_2019" # baseline loading column
fut_col = "median_annual_max_loading_rcp45hotter_2030_2059" # future loading column

at_risk_filter_column = hist_col # to filter population at-risk

save_folder = "all_regions"

TGW_scenario = "rcp45hotter"

TGW_weather_year = '2030_2059'


# cities to process
cities = ["AUS","GSO","SFO"]


CITY_REGIONS_TO_RUN = {
    "GSO": ["rural", "industrial", "urban-suburban"],
    "SFO": ["P1U", "P2U", "P1R"],
    "AUS": ["P1U", "P1R", "P2U"],

}

summary_save_dir = os.path.join(
    output_pf_path,
    save_folder,
    "summary_across_weather_years",
    TGW_scenario,
    solar_battery_scenario_folder,
)
transformers_summary_across_years_path = os.path.join(
    summary_save_dir,
    f"{transformers_file_name}_summary_across_weather_years.joblib",
)

lines_summary_across_years_path = os.path.join(
    summary_save_dir,
    f"{lines_file_name}_summary_across_weather_years.joblib",
)

## Load merged dictionary w/ summary of summary statistics across years

In [ ]:
# ============================================================
# Load period-level summary dictionaries
# ============================================================

# Check files exist before loading
for path in [
    transformers_summary_across_years_path,
    lines_summary_across_years_path,
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

summary_transformers_dict_weather = joblib.load(
    transformers_summary_across_years_path
)

summary_lines_dict_weather = joblib.load(
    lines_summary_across_years_path
)

print("Loaded:")
print(transformers_summary_across_years_path)
print(lines_summary_across_years_path)

# ============================================================
# Inspect loaded dictionaries
# ============================================================

print("\nTransformer summary dictionary nested keys:")
file_ops.print_nested_keys_structure(summary_transformers_dict_weather)

print("\nTransformer summary dictionary sample dataframe:")
file_ops.print_nested_dict_key_examples_and_dataframe_details(
    summary_transformers_dict_weather
)

## Process data (Concatenate regions, add KVA column, remove unused columns)

In [ ]:
### Concatenate regions to single city level dataframes (e.g., a single df with all transformers in GSO)
transformers_dict_summary_multi_region_agg_by_city, lines_dict_summary_multi_region_agg_by_city = df_ops.concat_regions_to_city(summary_transformers_dict_weather, summary_lines_dict_weather, TGW_weather_year, TGW_scenario, smart_ds_year, CITY_REGIONS_TO_RUN)

for city in cities:
    if 'kVA rating [kVA]' not in lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].columns:
        lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].insert(9, "kVA rating [kVA]",  mva_series_lines(lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]), allow_duplicates=False)

# List of columns to keep in transformers dataframes
col_to_keep_xfer = [
       'Transformer',
       'KV rating [kV]',
       'kVA rating [kVA]',
       hist_col,
       fut_col, 
       'region']

## --- Remove unused columns ---
for TGW_weather_year_scenario, city_dictionary in transformers_dict_summary_multi_region_agg_by_city.items():
    for smartds_year_city, df in city_dictionary.items():
        df_temp = city_dictionary[smartds_year_city]
        city_dictionary[smartds_year_city] = df_temp.loc[:, col_to_keep_xfer]

# List of columns to keep in lines dataframes
col_to_keep_lines = [
       'Line',
       'Nominal V [kV]',
       'kVA rating [kVA]',
       hist_col,
       fut_col, 
       'region']
        
for TGW_weather_year_scenario, city_dictionary in lines_dict_summary_multi_region_agg_by_city.items():
    for smartds_year_city, df in city_dictionary.items():
        df_temp = city_dictionary[smartds_year_city]
        city_dictionary[smartds_year_city] = df_temp.loc[:, col_to_keep_lines]

display(transformers_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].head(2))
display(lines_dict_summary_multi_region_agg_by_city[(TGW_weather_year, TGW_scenario)][(smart_ds_year, city)].head(2))

## Add asset upgrade cost

In [ ]:
start_time = time.time()

## Add column with upgrade cost
for TGW_weather_year_scenario, city_dictionary in transformers_dict_summary_multi_region_agg_by_city.items():

    for smartds_year_city, df in city_dictionary.items():

        # --- Transformers ---
        df_xfer = transformers_dict_summary_multi_region_agg_by_city[TGW_weather_year_scenario][smartds_year_city]

        df_xfer[[
            "Upgrade cost 25th ($/kW)",
            "Upgrade cost median ($/kW)",
            "Upgrade cost 75th ($/kW)"
        ]] = df_xfer["kVA rating [kVA]"].apply(
            lambda kva: get_upgrade_costs(kva, upgrade_type="substation")
        ).apply(pd.Series)

        display(df_xfer.head(2))

        # --- Lines ---
        df_lines = lines_dict_summary_multi_region_agg_by_city[TGW_weather_year_scenario][smartds_year_city]

        df_lines[[
            "Upgrade cost 25th ($/kW)",
            "Upgrade cost median ($/kW)",
            "Upgrade cost 75th ($/kW)"
        ]] = df_lines["kVA rating [kVA]"].apply(
            lambda kw: get_upgrade_costs(kw, upgrade_type="circuit")
        ).apply(pd.Series)

        display(df_lines.head(2))

display(df_xfer.head(2))
display(df_lines.head(2))

end_time = time.time(); print("Runtime:", (end_time - start_time) / 60, "minutes")

## Add delta capacity (includes both at-risk and critical)

In [ ]:
# Loading safety threhold to determine required capacity
xfm_th_cand = xfm_cand # e.g., 80%
line_th_cand = line_cand # e.g., 67%

xfm_th_crit = xfm_crit # e.g., 100%
line_th_crit = line_crit # e.g., 100%


for TGW_weather_year_scenario, city_dictionary in transformers_dict_summary_multi_region_agg_by_city.items():
    for smartds_year_city, df in city_dictionary.items():

        df_xfer = transformers_dict_summary_multi_region_agg_by_city[TGW_weather_year_scenario][smartds_year_city]

        ## Candidate loading thredhold for grid reinforcment 
        # Future Δ capacity needed
        df_xfer['cand F delta upgrade [kVA]'] = (
            df_xfer['kVA rating [kVA]'] * (df_xfer[fut_col] / xfm_th_cand - 1)
        ).clip(lower=0)

        # Baseline Δ capacity needed
        df_xfer['cand B delta upgrade [kVA]'] = (
            df_xfer['kVA rating [kVA]'] * (df_xfer[hist_col] / xfm_th_cand - 1)
        ).clip(lower=0)

        # Difference
        df_xfer['cand F_minus_B delta upgrade [kVA]'] = (
            df_xfer['cand F delta upgrade [kVA]'] - df_xfer['cand B delta upgrade [kVA]']
        )
        
        ## Critical loading thredhold for grid reinforcment 
        # Future Δ capacity needed
        df_xfer['crit F delta upgrade [kVA]'] = (
            df_xfer['kVA rating [kVA]'] * (df_xfer[fut_col] / xfm_th_crit - 1)
        ).clip(lower=0)

        # Baseline Δ capacity needed
        df_xfer['crit B delta upgrade [kVA]'] = (
            df_xfer['kVA rating [kVA]'] * (df_xfer[hist_col] / xfm_th_crit - 1)
        ).clip(lower=0)

        # Difference
        df_xfer['crit F_minus_B delta upgrade [kVA]'] = (
            df_xfer['crit F delta upgrade [kVA]'] - df_xfer['crit B delta upgrade [kVA]']
        )

for TGW_weather_year_scenario, city_dictionary in transformers_dict_summary_multi_region_agg_by_city.items():
    for smartds_year_city, df in city_dictionary.items():

        df_lines = lines_dict_summary_multi_region_agg_by_city[TGW_weather_year_scenario][smartds_year_city]

        ## Candidate loading thredhold for grid reinforcment 
        # Future Δ capacity needed
        df_lines['cand F delta upgrade [kVA]'] = (
            df_lines['kVA rating [kVA]'] * (df_lines[fut_col] / line_th_cand - 1)
        ).clip(lower=0)

        # Baseline Δ capacity needed
        df_lines['cand B delta upgrade [kVA]'] = (
            df_lines['kVA rating [kVA]'] * (df_lines[hist_col] / line_th_cand - 1)
        ).clip(lower=0)

        # Difference
        df_lines['cand F_minus_B delta upgrade [kVA]'] = (
            df_lines['cand F delta upgrade [kVA]'] - df_lines['cand B delta upgrade [kVA]']
        )
        
        ## Critical loading thredhold for grid reinforcment 
        # Future Δ capacity needed
        df_lines['crit F delta upgrade [kVA]'] = (
            df_lines['kVA rating [kVA]'] * (df_lines[fut_col] / line_th_crit - 1)
        ).clip(lower=0)

        # Baseline Δ capacity needed
        df_lines['crit B delta upgrade [kVA]'] = (
            df_lines['kVA rating [kVA]'] * (df_lines[hist_col] / line_th_crit - 1)
        ).clip(lower=0)

        # Difference
        df_lines['crit F_minus_B delta upgrade [kVA]'] = (
            df_lines['crit F delta upgrade [kVA]'] - df_lines['crit B delta upgrade [kVA]']
        )
        
display(df_xfer.head(2))
display(df_lines.head(2))

## Add age  

In [ ]:
# -----------------------------------------------------------
# add age column to each city DataFrame
# -----------------------------------------------------------

for TGW_weather_year_scenario, city_dictionary in transformers_dict_summary_multi_region_agg_by_city.items():
    for smartds_year_city, df in city_dictionary.items():

        df_xfer = transformers_dict_summary_multi_region_agg_by_city[
            TGW_weather_year_scenario
        ][smartds_year_city]

        # ---------------------------------------------------
        # Voltage class masks
        # ---------------------------------------------------

        kv_numeric = df_xfer["KV rating [kV]"].apply(extract_min_kv)

        xfer_mask_LV = kv_numeric < 1
        xfer_mask_MV = (kv_numeric >= 1) & (kv_numeric <= 35)
        xfer_mask_HV = kv_numeric > 35

        # ---------------------------------------------------
        # Initialize age column
        # ---------------------------------------------------
        df_xfer["Age (Years)"] = np.nan

        # ---------------------------------------------------
        # LV transformers: deterministic histogram matching
        # ---------------------------------------------------
        n_lv = xfer_mask_LV.sum()
        if n_lv > 0:
            lv_ages = assign_ages_deterministic(n_lv, age_df)

            # Assign in arbitrary but deterministic order (index order)
            df_xfer.loc[xfer_mask_LV, "Age (Years)"] = lv_ages

        # ---------------------------------------------------
        # MV / HV transformers: assign median age
        # ---------------------------------------------------
        df_xfer.loc[xfer_mask_MV | xfer_mask_HV, "Age (Years)"] = median_age

# -----------------------------------------------------------
#  add age column to line DataFrames
# -----------------------------------------------------------

for TGW_weather_year_scenario, city_dictionary in lines_dict_summary_multi_region_agg_by_city.items():
    for smartds_year_city, df in city_dictionary.items():

        df_line = lines_dict_summary_multi_region_agg_by_city[TGW_weather_year_scenario][smartds_year_city]

        # Voltage class masks (LINES)
        kv_numeric = df_line["Nominal V [kV]"].apply(extract_line_kv)

        line_mask_LV = kv_numeric < 1
        line_mask_MV = (kv_numeric >= 1) & (kv_numeric <= 35)
        line_mask_HV = kv_numeric > 35

        # Initialize age column
        df_line["Age (Years)"] = np.nan

        # LV lines: deterministic histogram matching
        n_lv = line_mask_LV.sum()
        if n_lv > 0:
            lv_ages = assign_ages_deterministic(n_lv, age_df)

            # Assign in deterministic index order
            df_line.loc[line_mask_LV, "Age (Years)"] = lv_ages

        # MV / HV lines: assign median age
        df_line.loc[line_mask_MV | line_mask_HV, "Age (Years)"] = median_age


# -----------------------------------------------------------
# 4. Validation: compare LV age distribution to original
# -----------------------------------------------------------

# Collect all LV transformer ages across all cities/scenarios
assigned_lv_ages = []

for _, city_dictionary in transformers_dict_summary_multi_region_agg_by_city.items():
    for _, df in city_dictionary.items():
        kv_numeric = df["KV rating [kV]"].apply(extract_min_kv)
        assigned_lv_ages.extend(df.loc[kv_numeric < 1, "Age (Years)"].values)

assigned_lv_ages = np.array(assigned_lv_ages)

# Empirical distribution of assigned ages
assigned_dist = (
    pd.Series(assigned_lv_ages)
    .value_counts(normalize=True)
    .sort_index()
)


# # -----------------------------------------------------------
# #  validation for LV line age distribution
# # -----------------------------------------------------------

# assigned_lv_line_ages = []

# for _, city_dictionary in lines_dict_summary_multi_region_agg_by_city.items():
#     for _, df in city_dictionary.items():
#         kv_numeric = df["Nominal V [kV]"].apply(extract_line_kv)
#         assigned_lv_line_ages.extend(df.loc[kv_numeric < 1, "Age (Years)"].values)

# assigned_lv_line_ages = np.array(assigned_lv_line_ages)

# assigned_line_dist = (
#     pd.Series(assigned_lv_line_ages)
#     .value_counts(normalize=True)
#     .sort_index()
# )


# -----------------------------------------------------------
# 5. Plot: original vs assigned LV age distribution
# -----------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    age_df["Age (Years)"],
    age_df["prob"],
    label="Original age distribution",
    linewidth=2
)

plt.plot(
    assigned_dist.index,
    assigned_dist.values,
    linestyle="--",
    label="Assigned LV age distribution",
    linewidth=2
)

plt.xlabel("Transformer age (years)")
plt.ylabel("Probability")
plt.title("Validation: Deterministic LV age assignment")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Tables of total cost -  reactive and proactive total cost for Cand/Crit LV/MV/HV/ALL combinations 
using either candidate loading threshold or critical loading threshold

In [ ]:
# ------------------------------------------------------------
# Output dictionaries
# ------------------------------------------------------------

lines_total_costs_range = {}
lines_total_costs_range[(TGW_weather_year, TGW_scenario)] = {}

transformers_total_costs_range = {}
transformers_total_costs_range[(TGW_weather_year, TGW_scenario)] = {}


# ------------------------------------------------------------
# Cost percentile columns
# ------------------------------------------------------------

PERCENTILES = {
    "25p": cost_25p_col,
    "50p": cost_med_col,
    "75p": cost_75p_col,
}


# ------------------------------------------------------------
# Voltage class definitions
# ------------------------------------------------------------

VOLTAGE_LEVELS_xfer = {
    "LV": lambda df: df["KV rating [kV]"].apply(extract_min_kv) < 1,
    "MV": lambda df: (
        (df["KV rating [kV]"].apply(extract_min_kv) >= 1) &
        (df["KV rating [kV]"].apply(extract_min_kv) <= 35)
    ),
    "HV": lambda df: df["KV rating [kV]"].apply(extract_min_kv) > 35,
}

VOLTAGE_LEVELS_lines = {
    "LV": lambda df: df["Nominal V [kV]"].apply(extract_line_kv) < 1,
    "MV": lambda df: (
        (df["Nominal V [kV]"].apply(extract_line_kv) >= 1) &
        (df["Nominal V [kV]"].apply(extract_line_kv) <= 35)
    ),
    "HV": lambda df: df["Nominal V [kV]"].apply(extract_line_kv) > 35,
}


# ------------------------------------------------------------
# Cost calculation
# ------------------------------------------------------------

def compute_threshold_costs(
    df,
    mask_crossing,
    threshold_name,
    cost_col,
    T_life=40,
):
    """
    Calculate reactive and proactive costs for assets that cross
    one reinforcement threshold.

    Parameters
    ----------
    df : pd.DataFrame
        Asset dataframe.

    mask_crossing : pd.Series
        True for assets with:
            historical loading < threshold
            future loading >= threshold

    threshold_name : {"cand", "crit"}
        Determines which F_minus_B delta-upgrade column is used.

    cost_col : str
        Unit cost column.

    T_life : float, optional
        Asset lifetime in years. Default = 40.

    Returns
    -------
    reactive_cost : float
    proactive_cost : float
    """

    delta_col = f"{threshold_name} F_minus_B delta upgrade [kVA]"

    # Remaining lifetime fraction for reactive replacement
    T_remaining = T_life - df.loc[mask_crossing, "Age (Years)"]
    life_frac = (T_remaining / T_life).clip(lower=0.0)

    # Reactive:
    # remaining value of existing asset + additional required capacity
    reactive_cost = (
        (
            life_frac * df.loc[mask_crossing, "kVA rating [kVA]"] +
            df.loc[mask_crossing, delta_col]
        ) *
        df.loc[mask_crossing, cost_col]
    ).sum()

    # Proactive:
    # only additional required capacity
    proactive_cost = (
        df.loc[mask_crossing, delta_col] *
        df.loc[mask_crossing, cost_col]
    ).sum()

    return reactive_cost, proactive_cost


# ------------------------------------------------------------
# Build one row: ALL, LV, MV, or HV
# ------------------------------------------------------------

def build_cost_row(
    df,
    metric_name,
    scope_mask,
    cand_th,
    crit_th,
):
    """
    Build one table row for a given voltage scope.

    Candidate and critical thresholds are evaluated independently.
    """

    row = {"Metric": metric_name}

    threshold_definitions = {
        "cand": cand_th,
        "crit": crit_th,
    }

    for threshold_name, threshold in threshold_definitions.items():

        # Assets that newly cross this specific threshold
        mask_crossing = (
            scope_mask &
            (df[hist_col] < threshold) &
            (df[fut_col] >= threshold)
        )

        for p, cost_col in PERCENTILES.items():

            reactive_cost, proactive_cost = compute_threshold_costs(
                df=df,
                mask_crossing=mask_crossing,
                threshold_name=threshold_name,
                cost_col=cost_col,
            )

            row[f"{threshold_name}_F_not_B_Reac_{p}"] = reactive_cost
            row[f"{threshold_name}_F_not_B_Pro_{p}"] = proactive_cost

    return row


# ------------------------------------------------------------
# Build complete city cost table
# ------------------------------------------------------------

def build_cost_table_for_df(
    df,
    cand_th,
    crit_th,
    voltage_levels,
):
    """
    Build cost table for one city.

    Rows:
        ALL
        LV
        MV
        HV

    Columns:
        candidate / critical threshold
        x reactive / proactive
        x 25p / 50p / 75p unit costs
    """

    rows = []

    # -------------------------
    # ALL voltage levels
    # -------------------------

    mask_all = pd.Series(True, index=df.index)

    rows.append(
        build_cost_row(
            df=df,
            metric_name="ALL",
            scope_mask=mask_all,
            cand_th=cand_th,
            crit_th=crit_th,
        )
    )

    # -------------------------
    # LV / MV / HV
    # -------------------------

    for voltage_level in ["LV", "MV", "HV"]:

        voltage_mask = voltage_levels[voltage_level](df)

        rows.append(
            build_cost_row(
                df=df,
                metric_name=voltage_level,
                scope_mask=voltage_mask,
                cand_th=cand_th,
                crit_th=crit_th,
            )
        )

    # Build final dataframe
    df_rows = pd.DataFrame(rows)

    # Round cost results
    numeric_cols = df_rows.select_dtypes(include="number").columns
    df_rows[numeric_cols] = df_rows[numeric_cols].round(2)

    return df_rows


# ============================================================
# Build transformer cost tables
# ============================================================

for city in cities:

    df_xfer = transformers_dict_summary_multi_region_agg_by_city[
        (TGW_weather_year, TGW_scenario)
    ][
        (smart_ds_year, city)
    ]

    transformers_total_costs_range[
        (TGW_weather_year, TGW_scenario)
    ][city] = build_cost_table_for_df(
        df=df_xfer,
        cand_th=xfm_th_cand,
        crit_th=xfm_th_crit,
        voltage_levels=VOLTAGE_LEVELS_xfer,
    )


# ============================================================
# Build line cost tables
# ============================================================

for city in cities:

    df_lines = lines_dict_summary_multi_region_agg_by_city[
        (TGW_weather_year, TGW_scenario)
    ][
        (smart_ds_year, city)
    ]

    lines_total_costs_range[
        (TGW_weather_year, TGW_scenario)
    ][city] = build_cost_table_for_df(
        df=df_lines,
        cand_th=line_th_cand,
        crit_th=line_th_crit,
        voltage_levels=VOLTAGE_LEVELS_lines,
    )


# ============================================================
# Display example tables
# ============================================================

city = "GSO"

print(
    f"Upgrade costs for transformers in {city} "
    f"(scenario {TGW_weather_year} {TGW_scenario}):\n"
)
display(
    transformers_total_costs_range[
        (TGW_weather_year, TGW_scenario)
    ][city]
)

print(
    f"Upgrade costs for lines in {city} "
    f"(scenario {TGW_weather_year} {TGW_scenario}):\n"
)
display(
    lines_total_costs_range[
        (TGW_weather_year, TGW_scenario)
    ][city]
)

## Tables of cost per rate payer per year

In [ ]:
# ============================================================
# Initialize output dictionaries
# ============================================================

lines_cost_per_customer_range = {}
lines_cost_per_customer_range[(TGW_weather_year, TGW_scenario)] = {}

transformers_cost_per_customer_range = {}
transformers_cost_per_customer_range[(TGW_weather_year, TGW_scenario)] = {}

def total_customers_included_regions(city):
    """
    Return the number of customers represented by the regions
    included in CITY_REGIONS_TO_RUN for a given city.
    """

    customer_dicts = {
        "AUS": customers_austin,
        "GSO": customers_gso,
        "SFO": customers_sf,
    }

    city = city.upper()

    if city not in customer_dicts:
        return np.nan

    # Normalize region names so, e.g., "rural" matches "Rural"
    customer_counts = {
        str(region).lower(): count
        for region, count in customer_dicts[city].items()
    }

    regions_included = CITY_REGIONS_TO_RUN[city]

    missing_regions = [
        region for region in regions_included
        if str(region).lower() not in customer_counts
    ]

    if missing_regions:
        raise KeyError(
            f"Missing customer counts for {city}: {missing_regions}"
        )

    return sum(
        customer_counts[str(region).lower()]
        for region in regions_included
    )


# ============================================================
# Convert entire dataframe -> annualized cost per customer
# ============================================================

def convert_df_to_cost_per_customer_range(
    df,
    customers,
    wacc=0.078,
    lifetime_years=40,
):
    """
    Converts all numeric cost columns from total investment cost ($)
    to annualized cost per customer ($/customer/year) using a
    Capital Recovery Factor (CRF).
    """

    df_out = df.copy()

    # Capital Recovery Factor
    crf = (
        wacc * (1 + wacc) ** lifetime_years
        / ((1 + wacc) ** lifetime_years - 1)
    )

    # Apply CRF-based annualization to all numeric columns
    num_cols = df.select_dtypes(include="number").columns

    for col in num_cols:
        df_out[col] = (df[col] * crf) / customers

    return df_out


# ============================================================
# Build lines_cost_per_customer_range
# ============================================================

for city in cities:

    customers = total_customers_included_regions(city)

    df_lines = lines_total_costs_range[
        (TGW_weather_year, TGW_scenario)
    ][city]

    df_lines_converted = convert_df_to_cost_per_customer_range(
        df_lines,
        customers,
    )

    lines_cost_per_customer_range[
        (TGW_weather_year, TGW_scenario)
    ][city] = df_lines_converted


# ============================================================
# Build transformers_cost_per_customer_range
# ============================================================

for city in cities:

    customers = total_customers_included_regions(city)

    df_xfer = transformers_total_costs_range[
        (TGW_weather_year, TGW_scenario)
    ][city]

    df_xfer_converted = convert_df_to_cost_per_customer_range(
        df_xfer,
        customers,
    )

    transformers_cost_per_customer_range[
        (TGW_weather_year, TGW_scenario)
    ][city] = df_xfer_converted


# ============================================================
# Combine line + transformer cost per customer
# ============================================================

line_n_xfer_cost_per_customer_range = {}
line_n_xfer_cost_per_customer_range[
    (TGW_weather_year, TGW_scenario)
] = {}

for city in cities:

    df_lines = lines_cost_per_customer_range[
        (TGW_weather_year, TGW_scenario)
    ][city]

    df_xfer = transformers_cost_per_customer_range[
        (TGW_weather_year, TGW_scenario)
    ][city]

    # Same structure; use lines table as base
    df_sum = df_lines.copy()

    # Sum numeric columns
    num_cols = df_lines.select_dtypes(include="number").columns
    df_sum[num_cols] = (
        df_lines[num_cols] +
        df_xfer[num_cols]
    )

    line_n_xfer_cost_per_customer_range[
        (TGW_weather_year, TGW_scenario)
    ][city] = df_sum


# ============================================================
# Display example
# ============================================================

city = "AUS"

print(
    f"Customers represented in {city}: "
    f"{total_customers_included_regions(city):,}"
)

print("\nCombined line + transformer cost per customer:")
display(
    line_n_xfer_cost_per_customer_range[
        (TGW_weather_year, TGW_scenario)
    ][city]
)

## Figure for main

### Set parameters

In [ ]:
# ============================================================
# HORIZONTAL DOT-AND-RANGE FIGURE
# Pre-settings + preprocessing
# ============================================================

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Patch
import numpy as np
import pandas as pd
import warnings


# ============================================================
# Matplotlib settings
# ============================================================

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]
mpl.rcParams["text.usetex"] = True


font_size_default = 10
mpl.rcParams.update({
    "font.size": font_size_default,
    "axes.labelsize": font_size_default,
    "xtick.labelsize": font_size_default,
    "ytick.labelsize": font_size_default,
    "legend.fontsize": font_size_default,
})

strategy_label_fontsize = font_size_default 
savings_label_fontsize = font_size_default 
legend_fontsize = font_size_default 


# ============================================================
# USER-DEFINED FIGURE SETTINGS
# ============================================================

# Figure dimensions
# figure_size = (10.5, 4.3)

figure_size = (7.2, 3.2)



# ============================================================
# SETTINGS for line between ref points and annotation
# ============================================================

# Show line connecting reactive and proactive reference points
show_savings_connector = False
show_annotation_in_total_costs = False
show_annotation_in_annualized_costs = False

# ------------------------------------------------------------
# Colors
# ------------------------------------------------------------

# DEFAULT:
# Neutral gray for reactive + blue for proactive
strategy_colors = {
    "Reac": "#D55E00",
    "Pro": "#0072B2",
}


# ------------------------------------------------------------
# Axis limits
#
# None = automatic
#
# Examples:
# dot_total_cost_xlim = (0, 2000)
# dot_per_customer_xlim = (0, 1000)
# ------------------------------------------------------------

dot_total_cost_xlim = (0, 2500)
dot_per_customer_xlim = None


# ------------------------------------------------------------
# Reactive / proactive vertical separation
# ------------------------------------------------------------

strategy_offset = 0.1

city_spacing = 0.5


# ------------------------------------------------------------
# Corridor appearance
# ------------------------------------------------------------

# Both sensitivity ranges use corridors of the same height.
# They are distinguished by opacity:
#
# darker corridor:
#     loading-threshold range at median unit costs
#
# lighter corridor:
#     loading-threshold + unit-cost range (P25--P75)

corridor_height = 0.10

criterion_corridor_alpha = 0.65
full_corridor_alpha = 0.18


# ------------------------------------------------------------
# Reference point
# ------------------------------------------------------------

reference_marker_size = 60
reference_marker_edgecolor = "white"
reference_marker_edgewidth = 0.4


# ------------------------------------------------------------
# Reactive-proactive savings connector
# ------------------------------------------------------------

savings_connector_color = "0.65"
savings_connector_linewidth = 0.9


# Horizontal distance between the right-most reference point
# and the "XX% lower" annotation, as a fraction of panel range
savings_label_offset_fraction = -0.02
savings_label_y_offset = -0.04


# ------------------------------------------------------------
# Labels / layout
# ------------------------------------------------------------


# Position of Reactive / Proactive labels in axes coordinates.
# Negative values put them to the left of the plotting area.
strategy_label_x = -0.025

# Push city names farther left so there is room for
# Reactive / Proactive labels between the city name and plot.
city_label_pad = 50

grid_alpha = 0.45


# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------

legend_y = 1.21


# ============================================================
# Data settings
# ============================================================

scenario_key = (
    TGW_weather_year,
    TGW_scenario,
)

plot_cities = [
    "SFO",
    "GSO",
    "AUS",
]

city_nice = {
    "AUS": "Austin",
    "GSO": "Greensboro",
    "SFO": "San Francisco",
}

### preprocessing

In [ ]:
# ============================================================
# Customer count
# ============================================================

def total_customers_plot(city):
    """
    Return number of customers in the regions actually included
    in CITY_REGIONS_TO_RUN.
    """

    customer_dicts = {
        "AUS": customers_austin,
        "GSO": customers_gso,
        "SFO": customers_sf,
    }

    city = city.upper()

    customer_counts = {
        str(region).lower(): count
        for region, count
        in customer_dicts[city].items()
    }

    regions_included = CITY_REGIONS_TO_RUN[city]

    missing_regions = [
        region
        for region in regions_included
        if str(region).lower()
        not in customer_counts
    ]

    if missing_regions:
        raise KeyError(
            f"Missing customer counts for "
            f"{city}: {missing_regions}"
        )

    return sum(
        customer_counts[
            str(region).lower()
        ]
        for region in regions_included
    )


# ============================================================
# Extract one cost value
# ============================================================

def get_cost_value(
    df,
    metric,
    threshold,
    strategy,
    percentile,
):
    """
    threshold:
        "cand" = at-risk loading criterion
        "crit" = critical loading criterion

    strategy:
        "Reac" or "Pro"

    percentile:
        "25p", "50p", "75p"
    """

    col = (
        f"{threshold}_F_not_B_"
        f"{strategy}_{percentile}"
    )

    selected = df.loc[
        df["Metric"] == metric,
        col,
    ]

    if len(selected) != 1:
        raise ValueError(
            f"Expected exactly one row for "
            f"Metric={metric!r}, "
            f"but found {len(selected)}."
        )

    return float(
        selected.iloc[0]
    )


# ============================================================
# Reference + sensitivity ranges
# ============================================================

def get_reference_and_ranges(
    df,
    strategy,
):
    """
    Calculate reference value and sensitivity bounds.

    REFERENCE ("realistic scenario best guess")
    ------------------------
    LV:
        critical loading criterion
        + P50 unit cost

    MV/HV:
        at-risk loading criterion
        + P50 unit cost


    LOADING-THRESHOLD RANGE
    -----------------------
    Lower:
        all assets critical + P50

    Upper:
        all assets at-risk + P50


    FULL SENSITIVITY RANGE
    ----------------------
    Lower:
        all assets critical + P25

    Upper:
        all assets at-risk + P75
    """

    # --------------------------------------------------------
    # Reference:
    # hybrid loading criterion + P50 unit costs
    # --------------------------------------------------------

    reference = (
        get_cost_value(
            df,
            "LV",
            "crit",
            strategy,
            "50p",
        )
        +
        get_cost_value(
            df,
            "MV",
            "cand",
            strategy,
            "50p",
        )
        +
        get_cost_value(
            df,
            "HV",
            "cand",
            strategy,
            "50p",
        )
    )


    # --------------------------------------------------------
    # Loading-threshold range
    # Unit costs held at P50
    # --------------------------------------------------------

    criterion_low = get_cost_value(
        df,
        "ALL",
        "crit",
        strategy,
        "50p",
    )

    criterion_high = get_cost_value(
        df,
        "ALL",
        "cand",
        strategy,
        "50p",
    )


    # --------------------------------------------------------
    # Full sensitivity envelope
    # --------------------------------------------------------

    full_low = get_cost_value(
        df,
        "ALL",
        "crit",
        strategy,
        "25p",
    )

    full_high = get_cost_value(
        df,
        "ALL",
        "cand",
        strategy,
        "75p",
    )


    return {
        "reference": reference,
        "criterion_low": criterion_low,
        "criterion_high": criterion_high,
        "full_low": full_low,
        "full_high": full_high,
    }


# ============================================================
# Add line + transformer statistics
# ============================================================

def add_stats(
    stats_1,
    stats_2,
    scale=1.0,
):

    return {
        key: (
            stats_1[key]
            + stats_2[key]
        ) / scale
        for key in stats_1
    }


# ============================================================
# Reactive -> proactive savings
# ============================================================

def get_savings(
    reac_stats,
    pro_stats,
):

    savings = (
        reac_stats["reference"]
        - pro_stats["reference"]
    )

    savings_pct = (
        100 * savings
        / reac_stats["reference"]
        if reac_stats["reference"] != 0
        else np.nan
    )

    return (
        savings,
        savings_pct,
    )


# ============================================================
# Extract values for all cities
# ============================================================

total_plot_stats = {
    "Reac": {},
    "Pro": {},
}

cppy_plot_stats = {
    "Reac": {},
    "Pro": {},
}


for city in plot_cities:

    # --------------------------------------------------------
    # Total investment costs
    # --------------------------------------------------------

    df_lines = (
        lines_total_costs_range[
            scenario_key
        ][city]
    )

    df_xfer = (
        transformers_total_costs_range[
            scenario_key
        ][city]
    )


    # --------------------------------------------------------
    # Annualized line + transformer cost/customer
    # --------------------------------------------------------

    df_cppy = (
        line_n_xfer_cost_per_customer_range[
            scenario_key
        ][city]
    )


    for strategy in [
        "Reac",
        "Pro",
    ]:

        line_stats = (
            get_reference_and_ranges(
                df_lines,
                strategy,
            )
        )

        xfer_stats = (
            get_reference_and_ranges(
                df_xfer,
                strategy,
            )
        )


        # ----------------------------------------------------
        # Total investment cost:
        # dollars -> million dollars
        # ----------------------------------------------------

        total_plot_stats[
            strategy
        ][city] = add_stats(
            line_stats,
            xfer_stats,
            scale=1e6,
        )


        # ----------------------------------------------------
        # Annualized cost/customer/year
        # ----------------------------------------------------

        cppy_plot_stats[
            strategy
        ][city] = (
            get_reference_and_ranges(
                df_cppy,
                strategy,
            )
        )


for data_name, stats_dict in [

    (
        "Total investment cost",
        total_plot_stats,
    ),

    (
        "Per-customer cost",
        cppy_plot_stats,
    ),
]:

    for strategy in [
        "Reac",
        "Pro",
    ]:

        for city in plot_cities:

            s = stats_dict[
                strategy
            ][city]

            expected_order = (
                s["full_low"]
                <= s["criterion_low"]
                <= s["reference"]
                <= s["criterion_high"]
                <= s["full_high"]
            )

            if not expected_order:
                warnings.warn(
                    f"Unexpected range ordering for "
                    f"{data_name}, {city}, "
                    f"{strategy}: {s}"
                )


# ============================================================
# City labels
# ============================================================


city_labels = [

    (
        f"{city_nice[city]}\n"
    )

    for city in plot_cities
]


# ============================================================
# Dataframe of values used in figure
# ============================================================

plot_number_rows = []

for city in plot_cities:

    for strategy in [
        "Reac",
        "Pro",
    ]:

        total_s = (
            total_plot_stats[
                strategy
            ][city]
        )

        cppy_s = (
            cppy_plot_stats[
                strategy
            ][city]
        )

        plot_number_rows.append({

            "City": city,
            "Strategy": strategy,

            "Total_M$_reference":
                total_s["reference"],

            "Total_M$_criterion_low":
                total_s["criterion_low"],

            "Total_M$_criterion_high":
                total_s["criterion_high"],

            "Total_M$_full_low":
                total_s["full_low"],

            "Total_M$_full_high":
                total_s["full_high"],

            "CPPY_reference":
                cppy_s["reference"],

            "CPPY_criterion_low":
                cppy_s["criterion_low"],

            "CPPY_criterion_high":
                cppy_s["criterion_high"],

            "CPPY_full_low":
                cppy_s["full_low"],

            "CPPY_full_high":
                cppy_s["full_high"],
        })


df_plot_numbers_hybrid = pd.DataFrame(
    plot_number_rows
)

display(
    df_plot_numbers_hybrid
)

### Plot 

In [ ]:
# ============================================================
# HORIZONTAL DOT-AND-RANGE FIGURE
# ============================================================

def draw_horizontal_dot_range(
    ax,
    y_pos,
    stats,
    color,
):
    """
    Draw one strategy estimate using cost corridors.

    Light corridor:
        critical + P25
        ->
        at-risk + P75

    Dark corridor:
        all-critical + P50
        ->
        all-at-risk + P50

    Dot:
        hybrid reference
        critical LV + at-risk MV/HV
        with P50 unit costs
    """


    # --------------------------------------------------------
    # Full sensitivity corridor
    # Loading threshold + unit cost
    # --------------------------------------------------------

    full_width = (
        stats["full_high"]
        - stats["full_low"]
    )

    ax.add_patch(
        Rectangle(
            (
                stats["full_low"],
                y_pos - corridor_height / 2,
            ),
            full_width,
            corridor_height,
            facecolor=color,
            edgecolor="none",
            alpha=full_corridor_alpha,
            zorder=2,
        )
    )


    # --------------------------------------------------------
    # Loading-threshold corridor
    # Median unit costs
    # --------------------------------------------------------

    criterion_width = (
        stats["criterion_high"]
        - stats["criterion_low"]
    )

    ax.add_patch(
        Rectangle(
            (
                stats["criterion_low"],
                y_pos - corridor_height / 2,
            ),
            criterion_width,
            corridor_height,
            facecolor=color,
            edgecolor="none",
            alpha=criterion_corridor_alpha,
            zorder=3,
        )
    )


    # --------------------------------------------------------
    # Hybrid reference point
    # --------------------------------------------------------

    ax.scatter(
        stats["reference"],
        y_pos,
        s=reference_marker_size,
        color=color,
        edgecolor=reference_marker_edgecolor,
        linewidth=reference_marker_edgewidth,
        zorder=4,
    )


# ============================================================
# Draw one complete panel
# ============================================================

def draw_dot_range_panel(
    ax,
    stats_dict,
    xlim=None,
    show_savings_annotation=True,
):

    y = np.arange(
        len(plot_cities)
    )

    y_reac = (
        y
        + strategy_offset
    )

    y_pro = (
        y
        - strategy_offset
    )


    # --------------------------------------------------------
    # Determine range for annotation offsets
    # --------------------------------------------------------

    all_values = []

    for city in plot_cities:

        for strategy in [
            "Reac",
            "Pro",
        ]:

            s = stats_dict[
                strategy
            ][city]

            all_values.extend([
                s["full_low"],
                s["full_high"],
            ])


    panel_span = (
        max(all_values)
        - min(all_values)
    )

    if panel_span == 0:
        panel_span = 1.0

    label_offset = (
        savings_label_offset_fraction
        * panel_span
    )


    # --------------------------------------------------------
    # Draw each city
    # --------------------------------------------------------

    for i, city in enumerate(
        plot_cities
    ):

        reac = (
            stats_dict[
                "Reac"
            ][city]
        )

        pro = (
            stats_dict[
                "Pro"
            ][city]
        )


        # ----------------------------------------------------
        # Reactive range
        # ----------------------------------------------------

        draw_horizontal_dot_range(
            ax,
            y_reac[i],
            reac,
            strategy_colors["Reac"],
        )


        # ----------------------------------------------------
        # Proactive range
        # ----------------------------------------------------

        draw_horizontal_dot_range(
            ax,
            y_pro[i],
            pro,
            strategy_colors["Pro"],
        )


        # ----------------------------------------------------
        # Thin connector between reference points
        # ----------------------------------------------------
        
        if show_savings_connector:

            ax.plot(
                [
                    reac["reference"],
                    pro["reference"],
                ],
                [
                    y_reac[i],
                    y_pro[i],
                ],
                color=savings_connector_color,
                linewidth=savings_connector_linewidth,
                zorder=1,
            )


        # ----------------------------------------------------
        # Savings annotation
        # ----------------------------------------------------
        if show_savings_annotation:

            _, savings_pct = (
                get_savings(
                    reac,
                    pro,
                )
            )

            annotation_x = (
                max(
                    reac["reference"],
                    pro["reference"],
                )
                + label_offset
            )

            annotation_y = y[i] + savings_label_y_offset


            ax.text(
                annotation_x,
                annotation_y,
                f"{savings_pct:.0f}\\% lower",
                ha="left",
                va="center",
                fontsize=savings_label_fontsize,
                color="0.25",
                zorder=5,
            )


    # --------------------------------------------------------
    # Axis formatting
    # --------------------------------------------------------

    ax.set_yticks(
        y,
        city_labels,
    )

    ax.invert_yaxis()

    ax.xaxis.grid(
        True,
        linestyle=":",
        alpha=grid_alpha,
    )

    ax.yaxis.grid(False)

    ax.set_axisbelow(True)

    if xlim is not None:
        ax.set_xlim(xlim)
    else:
        # Extra room for savings annotations
        ax.margins(x=0.08)


    return (
        y,
        y_reac,
        y_pro,
    )


# ============================================================
# Create figure
# ============================================================

fig, (
    ax1,
    ax2,
) = plt.subplots(

    1,
    2,

    figsize=figure_size,

    sharey=True,

    constrained_layout=True,
)


# ============================================================
# Panel a — total investment cost
# ============================================================

(
    y,
    y_reac,
    y_pro,
) = draw_dot_range_panel(

    ax1,

    total_plot_stats,

    xlim=dot_total_cost_xlim,
    
    show_savings_annotation=show_annotation_in_total_costs,

)


ax1.set_xlabel(
    "Total investment cost "
    "(million USD)"
)


# ============================================================
# Panel b — annualized cost per customer
# ============================================================

draw_dot_range_panel(

    ax2,

    cppy_plot_stats,

    xlim=dot_per_customer_xlim,
    
    show_savings_annotation=show_annotation_in_annualized_costs,

)


ax2.set_xlabel(
    "Annualized cost per customer "
    "(USD yr$^{-1}$)"
)


# ============================================================
# City labels
#
# Push city names left to create a column for
# Reactive / Proactive labels.
# ============================================================

ax1.tick_params(
    axis="y",
    pad=city_label_pad,
)


# ============================================================
# Direct Reactive / Proactive labels
#
# These replace the strategy legend.
# ============================================================

for i, city in enumerate(
    plot_cities
):

    ax1.text(
        strategy_label_x,
        y_reac[i],
        "Reactive",
        transform=ax1.get_yaxis_transform(),
        ha="right",
        va="center",
        fontsize=strategy_label_fontsize,
        color=strategy_colors["Reac"],
    )

    ax1.text(
        strategy_label_x,
        y_pro[i],
        "Proactive",
        transform=ax1.get_yaxis_transform(),
        ha="right",
        va="center",
        fontsize=strategy_label_fontsize,
        color=strategy_colors["Pro"],
    )


# ============================================================
# Panel letters
# ============================================================

ax1.text(
    -0.08,
    1.04,
    r"\textbf{a}",
    transform=ax1.transAxes,
    fontsize=13,
    va="top",
    ha="left",
)

ax2.text(
    -0.08,
    1.04,
    r"\textbf{b}",
    transform=ax2.transAxes,
    fontsize=13,
    va="top",
    ha="left",
)


# ============================================================
# Horizontal figure-level legend
# ============================================================

legend_color = "0.25"


reference_handle = Line2D(
    [0],
    [0],
    marker="o",
    linestyle="none",
    markerfacecolor=legend_color,
    markeredgecolor="white",
    markersize=7,
    label="Reference",
)


criterion_handle = Patch(
    facecolor=legend_color,
    edgecolor="none",
    alpha=criterion_corridor_alpha,
    label=(
        "Range across loading thresholds "
        "(P50 unit costs)"
    ),
)



full_range_handle = Patch(
    facecolor=legend_color,
    edgecolor="none",
    alpha=full_corridor_alpha,
    label=(
        "Range across both loading thresholds and unit-costs "
        "(P25--P75)"
    ),
)


fig.legend(
    handles=[
        criterion_handle,
        full_range_handle,
        reference_handle,
    ],
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        legend_y,
    ),
    ncol=1,
    frameon=False,
    fontsize=legend_fontsize,
    handlelength=2.2,
    columnspacing=1.8,
)


# ============================================================
# Save + show
# ============================================================
fig_path = f"figures/costs/barplots_CA_numbers/reactive_proactive_horizontal_dot_range_{TGW_weather_year}_{TGW_scenario}_{save_folder}.pdf"

# Create the directory structure if it doesn't exist
os.makedirs(os.path.dirname(fig_path), exist_ok=True)

plt.savefig(fig_path,
    dpi=600,
    bbox_inches="tight",
)

plt.show()

## Table for SI

In [ ]:
# ======================================================================
# Create tables of total investment costs and annualized cost/customer
# ======================================================================

import pandas as pd
import numpy as np


# ======================================================================
# Settings
# ======================================================================

city_order = {
    "AUS": 0,
    "GSO": 1,
    "SFO": 2,
}

CITY_MAP = {
    "AUS": "Austin",
    "GSO": "Greensboro",
    "SFO": "San Francisco",
}

cities_table = [
    "AUS",
    "GSO",
    "SFO",
]

scenario_key = (
    TGW_weather_year,
    TGW_scenario,
)

# Number of decimals shown in manuscript tables
total_cost_decimals = 1       # million USD
cppy_decimals = 1             # USD/customer/year


# ======================================================================
# Helper: extract one value
# ======================================================================

def get_table_cost_value(
    df,
    strategy,
    threshold,
    percentile,
    metric="ALL",
):
    """
    Extract one cost estimate from an Option 6 cost table.

    strategy:
        "Reac" or "Pro"

    threshold:
        "cand" = at-risk loading criterion
        "crit" = critical loading criterion

    percentile:
        "25p", "50p", "75p"
    """

    col = (
        f"{threshold}_F_not_B_"
        f"{strategy}_{percentile}"
    )

    selected = df.loc[
        df["Metric"] == metric,
        col,
    ]

    if len(selected) != 1:
        raise ValueError(
            f"Expected exactly one row for "
            f"Metric={metric!r}, but found {len(selected)}."
        )

    return float(selected.iloc[0])


# ======================================================================
# Helper: build one 13-column table
# ======================================================================

def build_cost_table(
    table_type,
):
    """
    table_type:
        "total" -> total investment cost, million USD
        "cppy"  -> annualized cost/customer/year
    """

    rows = []


    for city in cities_table:

        row = {
            "City": CITY_MAP.get(city, city),
        }


        # --------------------------------------------------------------
        # Retrieve source tables
        # --------------------------------------------------------------

        if table_type == "total":

            df_lines = (
                lines_total_costs_range[
                    scenario_key
                ][city]
            )

            df_xfer = (
                transformers_total_costs_range[
                    scenario_key
                ][city]
            )

        elif table_type == "cppy":

            df_cppy = (
                line_n_xfer_cost_per_customer_range[
                    scenario_key
                ][city]
            )

        else:

            raise ValueError(
                "table_type must be either 'total' or 'cppy'."
            )


        # --------------------------------------------------------------
        # All 12 combinations:
        #
        # strategy × loading criterion × unit-cost percentile
        # --------------------------------------------------------------

        for strategy, strategy_label in [
            ("Reac", "Reactive"),
            ("Pro", "Proactive"),
        ]:

            for threshold, threshold_label in [
                ("cand", "At-risk"),
                ("crit", "Critical"),
            ]:

                for percentile, percentile_label in [
                    ("25p", "P25"),
                    ("50p", "P50"),
                    ("75p", "P75"),
                ]:


                    if table_type == "total":

                        # Lines + transformers
                        value = (
                            get_table_cost_value(
                                df_lines,
                                strategy,
                                threshold,
                                percentile,
                            )
                            +
                            get_table_cost_value(
                                df_xfer,
                                strategy,
                                threshold,
                                percentile,
                            )
                        ) / 1e6   # USD -> million USD


                    else:

                        # Already annualized and combined
                        value = get_table_cost_value(
                            df_cppy,
                            strategy,
                            threshold,
                            percentile,
                        )


                    row[
                        (
                            strategy_label,
                            threshold_label,
                            percentile_label,
                        )
                    ] = value


        rows.append(row)


    # ==================================================================
    # Convert to DataFrame
    # ==================================================================

    df = pd.DataFrame(rows)


    # ------------------------------------------------------------------
    # Convert columns to a 3-level MultiIndex
    #
    # City occupies the first column;
    # remaining columns are grouped by:
    #
    # Strategy -> loading criterion -> unit-cost percentile
    # ------------------------------------------------------------------

    new_columns = []

    for col in df.columns:

        if col == "City":
            new_columns.append(
                ("City", "", "")
            )

        else:
            new_columns.append(col)


    df.columns = pd.MultiIndex.from_tuples(
        new_columns,
        names=[
            "Strategy",
            "Loading criterion",
            "Unit cost",
        ],
    )


    return df


# ======================================================================
# Build tables
# ======================================================================

total_cost_table = build_cost_table(
    table_type="total"
)

cppy_table = build_cost_table(
    table_type="cppy"
)


# ======================================================================
# Format tables for display
# ======================================================================

total_cost_display = total_cost_table.copy()
cppy_display = cppy_table.copy()


# Format numeric columns
for col in total_cost_display.columns:

    if col[0] != "City":

        total_cost_display[col] = (
            total_cost_display[col]
            .map(
                lambda x:
                f"{x:.{total_cost_decimals}f}"
            )
        )


for col in cppy_display.columns:

    if col[0] != "City":

        cppy_display[col] = (
            cppy_display[col]
            .map(
                lambda x:
                f"{x:.{cppy_decimals}f}"
            )
        )


# ======================================================================
# Display tables
# ======================================================================

print(
    "Total investment costs "
    "(million USD)"
)

display(
    total_cost_display
)


print(
    "Annualized cost per customer "
    "(USD/customer/year)"
)

display(
    cppy_display
)

## Convert to Latex

In [ ]:
# ======================================================================
# Create narrower publication-ready LaTeX tables
#
# Layout:
#
# City | Strategy | At-risk (P25, P50, P75)
#                 | Critical (P25, P50, P75)
#
# Each city has two rows:
#   Reactive
#   Proactive
# ======================================================================


def reshape_cost_table_for_latex(
    source_df,
    decimals=1,
):
    """
    Convert the wide 13-column cost table into a narrower format
    with Reactive/Proactive in rows and loading criterion in columns.

    Resulting columns:
        City
        Strategy
        At-risk: P25, P50, P75
        Critical: P25, P50, P75
    """

    rows = []

    for _, row in source_df.iterrows():

        city = row[("City", "", "")]

        for strategy in [
            "Reactive",
            "Proactive",
        ]:

            rows.append({
                "City": city,
                "Strategy": strategy,

                ("At-risk", "P25"):
                    row[(strategy, "At-risk", "P25")],

                ("At-risk", "P50"):
                    row[(strategy, "At-risk", "P50")],

                ("At-risk", "P75"):
                    row[(strategy, "At-risk", "P75")],

                ("Critical", "P25"):
                    row[(strategy, "Critical", "P25")],

                ("Critical", "P50"):
                    row[(strategy, "Critical", "P50")],

                ("Critical", "P75"):
                    row[(strategy, "Critical", "P75")],
            })


    df = pd.DataFrame(rows)


    # ------------------------------------------------------------------
    # Create hierarchical column headers
    # ------------------------------------------------------------------

    df.columns = pd.MultiIndex.from_tuples([
        ("City", ""),
        ("Strategy", ""),

        ("At-risk", "P25"),
        ("At-risk", "P50"),
        ("At-risk", "P75"),

        ("Critical", "P25"),
        ("Critical", "P50"),
        ("Critical", "P75"),
    ])


    # ------------------------------------------------------------------
    # Format numeric values
    # ------------------------------------------------------------------

    for col in df.columns:

        if col[0] not in [
            "City",
            "Strategy",
        ]:

            df[col] = df[col].map(
                lambda x: f"{x:.{decimals}f}"
            )


    return df


# ======================================================================
# TOTAL INVESTMENT COST TABLE
# ======================================================================

latex_total_narrow = reshape_cost_table_for_latex(
    total_cost_table,
    decimals=total_cost_decimals,
)


# Use City + Strategy as hierarchical row index
latex_total_narrow = latex_total_narrow.set_index([
    ("City", ""),
    ("Strategy", ""),
])

latex_total_narrow.index.names = [
    "City",
    "Strategy",
]


latex_total_table = latex_total_narrow.to_latex(
    index=True,
    escape=False,
    sparsify=True,
    multirow=True,
    multicolumn=True,
    multicolumn_format="c",
    na_rep="--",

    # 2 text columns + 6 numeric columns
    column_format="llrrrrrr",

    position="htbp",

    caption=(
        "caption"
    ),

    label="tab:grid_reinforcement_total_cost_sensitivity",
)


print(latex_total_table)



# ======================================================================
# ANNUALIZED COST PER CUSTOMER TABLE
# ======================================================================

latex_cppy_narrow = reshape_cost_table_for_latex(
    cppy_table,
    decimals=cppy_decimals,
)


latex_cppy_narrow = latex_cppy_narrow.set_index([
    ("City", ""),
    ("Strategy", ""),
])

latex_cppy_narrow.index.names = [
    "City",
    "Strategy",
]


latex_cppy_table = latex_cppy_narrow.to_latex(
    index=True,
    escape=False,
    sparsify=True,
    multirow=True,
    multicolumn=True,
    multicolumn_format="c",
    na_rep="--",

    column_format="llrrrrrr",

    position="htbp",

    caption=(
        "caption."
    ),

    label="tab:grid_reinforcement_annualized_cost_sensitivity",
)


print(latex_cppy_table)